# Analyse Bayesienne 

## Ajout des bibliothèques nécessaires

In [76]:
#importation des données
import os
import glob

# Manipulation des données
import pandas as pd
import numpy as np

# Outils de Machine Learning (ici je vais utiliser Scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

print("Toutes les bibliothèques sont chargées avec succès !")

Toutes les bibliothèques sont chargées avec succès !


In [77]:
import os
import glob
import pandas as pd

def segmenter_en_paquets(texte, taille_paquet=10):
    phrases = texte.splitlines()
    phrases = [p.strip() for p in phrases if p.strip()]

    paquets = []
    for i in range(0, len(phrases), taille_paquet):
        paquet = " ".join(phrases[i:i+taille_paquet])
        paquets.append(paquet)

    return paquets


chemin_dossier = "N_vs_A"

liste_fichiers = sorted(glob.glob(os.path.join(chemin_dossier, "*.txt")))

print(f"Trouvé {len(liste_fichiers)} fichiers.")

# -----------------------------
# Séparation PAR ŒUVRE
# -----------------------------
import random

random.seed(1)
random.shuffle(liste_fichiers)
liste_train = liste_fichiers[:40]
liste_test = liste_fichiers[40:]

print(f"Fichiers train : {len(liste_train)}")
print(f"Fichiers test  : {len(liste_test)}")


def construire_dataframe(liste_fichiers):

    donnees = []

    for chemin_fichier in liste_fichiers:

        nom_fichier = os.path.basename(chemin_fichier)

        if nom_fichier.endswith("clean.txt"):
            label = "Zola"
        else:
            label = "naturaliste"

        with open(chemin_fichier, "r", encoding="utf-8") as f:
            texte = f.read()

        blocs = segmenter_en_paquets(texte)

        for bloc in blocs:
            donnees.append({
                "texte": bloc,
                "label": label,
                "source": nom_fichier
            })

    return pd.DataFrame(donnees)


df_train = construire_dataframe(liste_train)
df_test = construire_dataframe(liste_test)

print("-"*30)
print("TRAIN")
print(df_train["label"].value_counts())

print("-"*30)
print("TEST")
print(df_test["label"].value_counts())

Trouvé 49 fichiers.
Fichiers train : 40
Fichiers test  : 9
------------------------------
TRAIN
label
Zola           11050
naturaliste     7656
Name: count, dtype: int64
------------------------------
TEST
label
Zola           2857
naturaliste    1520
Name: count, dtype: int64


In [ ]:
print("Textes dans le train")
print(len(df_train))

sources_par_label = (
    df_train.groupby("label")["source"]
      .apply(lambda x: x.dropna().unique().tolist())
)

print("-" * 30)
print("Sources par label :")

for label, sources in sources_par_label.items():
    print(f"\n{label} :")
    for source in sources:
        print(f"  - {source}")
        
print("\n\n\n\nTextes dans le test)")
print(len(df_test))
        
sources_par_label = (
    df_test.groupby("label")["source"]
      .apply(lambda x: x.dropna().unique().tolist())
)

print("-" * 30)
print("Sources par label :")

for label, sources in sources_par_label.items():
    print(f"\n{label} :")
    for source in sources:
        print(f"  - {source}")

In [83]:
df_train


,texte,label,source
0,"Au milieu du grand silence, et dans le désert ...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
1,"cria un des hommes, qui s’était mis à genoux s...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
2,"Il regardait Mme François d’un air effaré, san...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
3,La maraîchère le vit qui s’appuyait en chancel...,Zola,1873_3_Le_ventre_de_Paris._clean.txt
4,"Les autres voitures suivirent, la file reprit ...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
...,...,...,...
18701,"mon vieux Claude, grand cœur d’enfant, tu sera...",Zola,1886_14_L_oeuvre._clean.txt
18702,"Revertitur in terram suam unde erat..., récita...",Zola,1886_14_L_oeuvre._clean.txt
18703,"Il avait tellement plu, les jours précédents, ...",Zola,1886_14_L_oeuvre._clean.txt
18704,"Il est bien heureux, dit Bongrand, il n’a pas ...",Zola,1886_14_L_oeuvre._clean.txt


In [84]:
df_test

,texte,label,source
0,I – Un article ?… Tu me demandes s’il y a un a...,naturaliste,Charles Demailly.txt
1,L’autre était un jeune homme de trente-quatre ...,naturaliste,Charles Demailly.txt
2,"Ces cinq hommes, qui avaient l’oreille à la bo...",naturaliste,Charles Demailly.txt
3,"C’était un garçon qui, sans façon, d’un bond, ...",naturaliste,Charles Demailly.txt
4,Tout conspirait donc pour la fortune du petit ...,naturaliste,Charles Demailly.txt
...,...,...,...
4372,C’était Bismarck qu’on allait reconduire chez ...,Zola,1880_9_Nana._clean.txt
4373,"Étonnée, Gaga ouvrit la porte, disparut un ins...",Zola,1880_9_Nana._clean.txt
4374,"toutes sortes d’idées, une envie d’y passer mo...",Zola,1880_9_Nana._clean.txt
4375,"elle est changée, elle est changée, murmurait ...",Zola,1880_9_Nana._clean.txt


In [80]:
X_train = df_train["texte"]
y_train = df_train["label"]

X_test = df_test["texte"]
y_test = df_test["label"]



modele_bayes = make_pipeline(
    TfidfVectorizer(
        max_features=10000,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9
    ),
    MultinomialNB(alpha=1.0)
)

modele_bayes.fit(X_train, y_train)

print("Modèle entraîné.")

Modèle entraîné.


In [81]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = modele_bayes.predict(X_test)

print(classification_report(y_test, predictions))

cm = confusion_matrix(y_test, predictions)
print(cm)

              precision    recall  f1-score   support

        Zola       0.95      0.97      0.96      2857
 naturaliste       0.94      0.91      0.93      1520

    accuracy                           0.95      4377
   macro avg       0.95      0.94      0.95      4377
weighted avg       0.95      0.95      0.95      4377

[[2772   85]
 [ 131 1389]]


In [82]:
cm = confusion_matrix(y_test, predictions)
print(cm)

[[2772   85]
 [ 131 1389]]


In [90]:
import numpy as np
import pandas as pd

vectorizer = modele_bayes.named_steps["tfidfvectorizer"]
classifieur = modele_bayes.named_steps["multinomialnb"]

mots = vectorizer.get_feature_names_out()
classes = classifieur.classes_

print("Ordre des classes :", classes)

# Différence des log-probabilités entre les deux classes
score_discriminant = (
    classifieur.feature_log_prob_[0]
    - classifieur.feature_log_prob_[1]
)

resultats_mots = pd.DataFrame({
    "mot": mots,
    f"score_{classes[0]}": score_discriminant
})

# Mots les plus caractéristiques de la première classe
mots_classe_0 = resultats_mots.sort_values(
    f"score_{classes[0]}",
    ascending=False
).head(30)

# Mots les plus caractéristiques de la deuxième classe
mots_classe_1 = resultats_mots.sort_values(
    f"score_{classes[0]}",
    ascending=True
).head(30)

print(f"\nMots les plus caractéristiques de {classes[0]} :")
for _, ligne in mots_classe_0.iterrows():
    print(f"{ligne['mot']:<25} {ligne[f'score_{classes[0]}']:.3f}")

print(f"\nMots les plus caractéristiques de {classes[1]} :")
for _, ligne in mots_classe_1.iterrows():
    print(f"{ligne['mot']:<25} {-ligne[f'score_{classes[0]}']:.3f}")

Ordre des classes : ['Zola' 'naturaliste']

Mots les plus caractéristiques de Zola :
rougon                    4.148
mouret                    3.805
gervaise                  3.761
saccard                   3.727
coupeau                   3.633
pauline                   3.626
denise                    3.464
étienne                   3.439
octave                    3.417
buteau                    3.406
florent                   3.305
faujas                    3.295
chanteau                  3.199
plassans                  3.181
lisa                      3.173
claude                    3.164
josserand                 3.143
maurice                   3.088
renée                     3.081
silvère                   3.032
abbé faujas               3.026
clorinde                  2.960
maheu                     2.925
lantier                   2.924
lorilleux                 2.889
macquart                  2.878
fouan                     2.876
sandoz                    2.826
duveyrier          

In [91]:
for mot in ["gervaise", "étienne", "denise", "pauline", "coupeau"]:
    nb = df_test["texte"].str.contains(mot, case=False).sum()
    print(mot, nb)

gervaise 1
étienne 5
denise 1
pauline 70
coupeau 0


In [92]:
import numpy as np
import pandas as pd

# Récupération des étapes du pipeline
vectorizer = modele_bayes.named_steps["tfidfvectorizer"]
classifieur = modele_bayes.named_steps["multinomialnb"]

# Vocabulaire appris sur le train
termes = vectorizer.get_feature_names_out()

# Ordre réel des classes
classes = classifieur.classes_
print("Ordre des classes :", classes)

# Transformation du jeu de test avec le vectorizer déjà entraîné
X_test_tfidf = vectorizer.transform(df_test["texte"])

# Nombre d'extraits du test contenant chaque terme
presence_test = np.asarray((X_test_tfidf > 0).sum(axis=0)).ravel()

# Somme des poids TF-IDF de chaque terme dans le test
poids_tfidf_test = np.asarray(X_test_tfidf.sum(axis=0)).ravel()

# Score discriminant appris par Naive Bayes
# score positif : classe 0
# score négatif : classe 1
score_discriminant = (
    classifieur.feature_log_prob_[0]
    - classifieur.feature_log_prob_[1]
)

resultats = pd.DataFrame({
    "terme": termes,
    "score_discriminant": score_discriminant,
    "nb_extraits_test": presence_test,
    "poids_tfidf_test": poids_tfidf_test
})

# On conserve uniquement les termes présents dans le test
resultats_test = resultats[
    resultats["nb_extraits_test"] > 0
].copy()

print(f"\nNombre de termes du vocabulaire présents dans le test : "
      f"{len(resultats_test)}")

Ordre des classes : ['Zola' 'naturaliste']

Nombre de termes du vocabulaire présents dans le test : 9709


In [93]:
n = 30

termes_classe_0 = (
    resultats_test
    .sort_values("score_discriminant", ascending=False)
    .head(n)
)

termes_classe_1 = (
    resultats_test
    .sort_values("score_discriminant", ascending=True)
    .head(n)
)

print(f"\nTermes présents dans le test les plus caractéristiques de {classes[0]} :")

for _, ligne in termes_classe_0.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"score={ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )

print(f"\nTermes présents dans le test les plus caractéristiques de {classes[1]} :")

for _, ligne in termes_classe_1.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"score={-ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )


Termes présents dans le test les plus caractéristiques de Zola :
rougon                    score=4.148  extraits=3
mouret                    score=3.805  extraits=157
gervaise                  score=3.761  extraits=1
pauline                   score=3.626  extraits=70
denise                    score=3.464  extraits=1
étienne                   score=3.439  extraits=1
octave                    score=3.417  extraits=2
plassans                  score=3.181  extraits=14
lisa                      score=3.173  extraits=15
claude                    score=3.164  extraits=2
maurice                   score=3.088  extraits=2
lantier                   score=2.924  extraits=9
macquart                  score=2.878  extraits=1
lazare                    score=2.807  extraits=13
boche                     score=2.748  extraits=1
maxime                    score=2.729  extraits=20
miette                    score=2.720  extraits=3
empereur                  score=2.670  extraits=17
prussiens                 

In [95]:
resultats_test["contribution_test"] = (
    resultats_test["score_discriminant"]
    * resultats_test["poids_tfidf_test"]
)

In [96]:
contributions_zola = (
    resultats_test
    .sort_values("contribution_test", ascending=False)
    .head(30)
)

contributions_naturalistes = (
    resultats_test
    .sort_values("contribution_test", ascending=True)
    .head(30)
)

print(f"\nTermes ayant le plus contribué aux prédictions {classes[0]} :")

for _, ligne in contributions_zola.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"contribution={ligne['contribution_test']:.3f}  "
        f"score={ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )

print(f"\nTermes ayant le plus contribué aux prédictions {classes[1]} :")

for _, ligne in contributions_naturalistes.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"contribution={-ligne['contribution_test']:.3f}  "
        f"score={-ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )


Termes ayant le plus contribué aux prédictions Zola :
nana                      contribution=237.834  score=2.545  extraits=539
qu                        contribution=160.484  score=1.263  extraits=2276
elle                      contribution=123.174  score=0.401  extraits=3162
était                     contribution=86.348  score=0.730  extraits=2663
eugène                    contribution=75.551  score=1.766  extraits=255
qu il                     contribution=75.173  score=1.260  extraits=1178
qu elle                   contribution=74.754  score=1.385  extraits=900
avait                     contribution=66.151  score=0.593  extraits=2541
ça                        contribution=63.921  score=0.991  extraits=975
lorsqu                    contribution=57.187  score=2.429  extraits=388
abbé                      contribution=55.610  score=2.476  extraits=236
mouret                    contribution=50.558  score=3.805  extraits=157
une                       contribution=47.441  score=0.252  e